<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 19 · Signals, Forecasts, and Portfolio Implementation
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the main Chapter 19 examples:
- build simple momentum-based features and forward-return targets,
- compute daily information coefficients for a signal,
- map standardised signals to long-only portfolio weights, and
- explore how rebalancing frequency affects turnover.


### Imports
We start with core scientific Python libraries and set Matplotlib defaults to
match the book style.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
# Matplotlib defaults for the book
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif"})
mpl.rcParams.update({"figure.dpi": 300})

### Data Preparation and Features
Load the end-of-day dataset, compute daily returns for a small universe, and
construct simple 20-day momentum features.


In [ ]:
LOCAL_EOD = Path("..") / "data" / "eod_data.csv"
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)

universe = ["AAPL", "JPM", "TLT"]
cols = universe + ["SPY"]
sub = prices[cols].dropna(how="any")

rets = sub[universe].pct_change().dropna()
rets = rets.iloc[-2 * 252 :]

mom_20d = rets.rolling(20).mean()
mom_20d.tail()

### Targets and Information Coefficients
Define simple 5-day forward-return targets and compute a daily Spearman
information coefficient (IC) between momentum and subsequent returns.


In [ ]:
fwd_5d = rets.shift(-5).rolling(5).sum()
aligned_index = mom_20d.index.intersection(fwd_5d.index)
mom_aligned = mom_20d.loc[aligned_index]
fwd_aligned = fwd_5d.loc[aligned_index]

def daily_ic(signal, target):
    """Daily Spearman rank IC time series."""
    rows = []
    for date, x in signal.iterrows():
        y = target.loc[date]
        if x.isna().any() or y.isna().any():
            continue
        rows.append(x.rank().corr(y, method="spearman"))
    return pd.Series(rows, name="ic")

ic = daily_ic(mom_aligned, fwd_aligned)
ic.describe().round(3)

### From Signals to Long-Only Weights
Take the latest momentum values, standardise them, and map the positive
z-scores to long-only portfolio weights.


In [ ]:
latest = mom_20d.iloc[-1]
z_scores = (latest - latest.mean()) / latest.std()
z_pos = z_scores.clip(lower=0)
if z_pos.sum() == 0:
    w_latest = pd.Series(
        np.repeat(1.0 / len(universe), len(universe)),
        index=universe,
    )
else:
    w_latest = (z_pos / z_pos.sum()).reindex(universe)

z_scores.reindex(universe), w_latest

### Weekly Rebalancing and Turnover
Recompute signal-based weights weekly, hold them daily, and estimate the
resulting annualised turnover.


In [ ]:
weekly = mom_20d.resample("W-FRI").last().dropna()

def weights_from_row(row):
    z = (row - row.mean()) / row.std()
    z_pos = z.clip(lower=0)
    if z_pos.sum() == 0:
        return np.repeat(1.0 / len(row), len(row))
    return (z_pos / z_pos.sum()).values

w_weekly = weekly.apply(
    weights_from_row,
    axis=1,
    result_type="expand",
)
w_weekly.columns = universe
w_weekly.head()

In [ ]:
w_daily = w_weekly.reindex(rets.index, method="ffill")

def turnover_series(weights):
    diff = weights.diff().abs().sum(axis=1)
    return 0.5 * diff

to_daily = turnover_series(w_daily)
to_annual = float(to_daily.mean() * 252.0)
to_annual

### Figures
The next cells reproduce the chapter figures for mapping ranks to weights
and for comparing turnover under weekly and monthly rebalancing.


In [ ]:
fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(6.4, 5.2),
    sharex=True,
)

x = np.arange(len(universe))
width = 0.35

ax1.bar(x, z_scores.reindex(universe).values, width)
ax1.axhline(0.0, color="black", linewidth=0.8, alpha=0.8)
ax1.set_ylabel("Momentum z-score")
ax1.set_title("Standardised signals")
ax1.grid(True, axis="y", linestyle="--", alpha=0.3)

ax2.bar(x, w_latest.values, width)
ax2.set_xticks(x)
ax2.set_xticklabels(universe)
ax2.set_ylabel("Portfolio weight")
ax2.set_title("Long-only weights from positive z-scores")
ax2.grid(True, axis="y", linestyle="--", alpha=0.3)

fig.tight_layout()

In [ ]:
mom_monthly = mom_20d.resample("ME").last().dropna()
w_monthly = mom_monthly.apply(
    weights_from_row,
    axis=1,
    result_type="expand",
)
w_monthly.columns = universe
w_monthly = w_monthly.reindex(rets.index, method="ffill")

w_weekly_daily = w_weekly.reindex(rets.index, method="ffill")
to_weekly = float(turnover_series(w_weekly_daily).mean() * 252.0)
to_monthly = float(turnover_series(w_monthly).mean() * 252.0)

freqs = ["Weekly", "Monthly"]
values = [to_weekly, to_monthly]
x = np.arange(len(freqs))

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.bar(x, values, width=0.6, color=["tab:blue", "tab:orange"])
ax.set_xticks(x)
ax.set_xticklabels(freqs)
ax.set_ylabel("Annualised turnover")
ax.set_title("Turnover under different rebalancing frequencies")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)

for xpos, val in zip(x, values):
    ax.text(xpos, val, f"{val:.2f}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
